# 02 — Modelling Dataset Preparation and Feature Engineering

This notebook prepares modelling inputs from the cleaned transaction dataset produced in the data exploration stage. The dataset contains 130 products selected for sustained demand activity during the training period. Preparation initially focuses on one product before extending the same procedure to the remaining products.

The forecasting target is next-day sales, used as a proxy for demand. Each prediction uses only information available by the end of the current day, together with calendar information known in advance.

Two input formats are prepared. Chronologically ordered daily sales series support the seasonal naïve baseline and SARIMA models, and can subsequently be organised into fixed-length sequences for recurrent neural networks. Tabular datasets support models such as linear regression, Random Forest, and XGBoost. These datasets include historical sales lags, including the corresponding day of the previous week, rolling means and standard deviations over 7- and 28-day windows, and the day of the week of the forecast date. All historical features exclude the target day's sales to prevent information leakage.

The resulting datasets provide consistent inputs for subsequent model training and chronological validation. Forecasting models will be compared against the baselines, and the best-performing model on the validation set will be selected for the subsequent forecasting and inventory simulation stages.

In [1]:
# Configure AWS login and input location
import boto3
from sagemaker.core.helper.session_helper import Session

sagemaker_session = Session()

REGION = sagemaker_session.boto_region_name
BUCKET = "smart-stock-s3-bucket"

DATASET_KEY = "smart-stock/data/processed/selected_product_transactions.csv"
TRANSACTIONS_DATA_S3_URI = f"s3://{BUCKET}/{DATASET_KEY}"

print("Region:", REGION)
print("Dataset location:", TRANSACTIONS_DATA_S3_URI) 

Region: eu-north-1
Dataset location: s3://smart-stock-s3-bucket/smart-stock/data/processed/selected_product_transactions.csv


In [2]:
# import relevant libraries and load processed dataset
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
from statsmodels.graphics.tsaplots import plot_acf

df = pd.read_csv(TRANSACTIONS_DATA_S3_URI)

In [3]:
# quick overview about the dataset
print("Number of rows and columns:", df.shape)
print("\n Column names: ")
print(df.columns.tolist())

display(df.head())
display(df.info())

Number of rows and columns: (125403, 8)

 Column names: 
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom
1,489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01 09:06:00,1.69,13078.0,United Kingdom
2,489437,84970S,HANGING HEART ZINC T-LIGHT HOLDER,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom
3,489437,20971,PINK BLUE FELT CRAFT TRINKET BOX,12,2009-12-01 09:08:00,1.25,15362.0,United Kingdom
4,489437,21912,VINTAGE SNAKES & LADDERS,4,2009-12-01 09:08:00,3.75,15362.0,United Kingdom


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125403 entries, 0 to 125402
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      125403 non-null  int64  
 1   StockCode    125403 non-null  object 
 2   Description  125403 non-null  object 
 3   Quantity     125403 non-null  int64  
 4   InvoiceDate  125403 non-null  object 
 5   Price        125403 non-null  float64
 6   Customer ID  100471 non-null  float64
 7   Country      125403 non-null  object 
dtypes: float64(2), int64(2), object(4)
memory usage: 7.7+ MB


None

In [4]:
df["StockCode"].nunique()

130

## Dataset transformation
We need to transform the dataset into a modelling input. Only three columns are needed for the modelling task. Invoice date provides the time of the transactions and serves as index of the modelling dataset. StockCode is used as columns because each product has its own sale series over time and Quantity is the number of item sold on a particular day. The aggregation and transformation is made in the following code chunk.

In [6]:
# Make a multivariate daily sales series where unavailable transactions 
# are zero sales -- sales are proxy for demand, so the dataset is called daily_demand
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
daily_demand = (df
    .assign(date=df["InvoiceDate"]
    .dt.normalize()).groupby(["date", "StockCode"])["Quantity"]
    .sum()
    .unstack("StockCode", fill_value=0)
    .sort_index())

# Include calendar days with no transactions across all selected products.
calendar = pd.date_range(
    daily_demand.index.min(),
    daily_demand.index.max(),
    freq="D",
)

daily_demand = daily_demand.reindex(calendar, fill_value=0)
daily_demand.index.name = "date"

In [8]:
daily_demand.shape

(739, 130)

The new demand series contains 739 days sapnning the entire transactions record window. The 130 columns represent the selected 130 products that have sufficient demand activity across the (training) window. Now let's add some calendar features before saving the sequence dataset.

In [9]:
# Since rows contain observed demand through the current day,
# create a copy of the demand dataset and add the calendar feature
# for the next day, which is the target. 
multivariate_inputs = daily_demand.copy()

# Calendar information for the next day being forecast.
target_dates = daily_demand.index + pd.Timedelta(days=1)

multivariate_inputs["target_weekday"] = target_dates.dayofweek
multivariate_inputs["target_month"] = target_dates.month
multivariate_inputs["target_is_weekend"] = (
    target_dates.dayofweek >= 5 # pandas counts days from 0 to 6
).astype(int)

In [10]:
multivariate_inputs.head()

StockCode,15036,20674,20675,20676,20677,20685,20725,20726,20727,20728,...,85040A,85049A,85049D,85049E,85099B,85123A,85152,target_weekday,target_month,target_is_weekend
date,,,,,,,,,,,,,,,,,,,,,
2009-12-01,55,1,16,17,16,47,164,34,23,44,...,1,31,14,19,164,261,24,2,12,0
2009-12-02,15,72,88,26,75,20,12,25,77,5,...,0,45,1,34,246,750,0,3,12,0
2009-12-03,243,0,20,26,15,24,31,17,32,41,...,6,3,0,12,143,359,64,4,12,0
2009-12-04,12,0,8,8,1,38,73,31,72,56,...,8,9,0,5,587,179,22,5,12,1
2009-12-05,12,0,0,0,0,12,0,0,10,0,...,0,0,0,0,40,96,6,6,12,1


In [11]:
# save the sequence datasets (product features only and product features with calendar features)
daily_demand.to_csv(
    f"s3://{BUCKET}/smart-stock/data/modelling/daily_demand.csv",
    index=True,
    index_label="date",
)

multivariate_inputs.to_csv(
    f"s3://{BUCKET}/smart-stock/data/modelling/multivariate_inputs.csv",
    index=True,
    index_label="date",
)

## Feature engineering 
Select one product and make an engineered feature set, on row per forecast date 

In [12]:
# one selected product
PRODUCT_ID = "84946"

# Extract the selected product's daily demand.
sales = daily_demand.copy()
sales.columns = sales.columns.astype(str)

daily_sales = sales[PRODUCT_ID].sort_index()

features_df = daily_sales.rename("target_sales").to_frame()
features_df.index.name = "forecast_date"

# Historical demand relative to the forecast date.
for lag in [1, 7, 14, 28]:
    features_df[f"sales_lag_{lag}"] = daily_sales.shift(lag)

# Shift first to exclude the target day's demand.
past_sales = daily_sales.shift(1)

for window in [7, 28]:
    features_df[f"sales_mean_{window}"] = (
        past_sales.rolling(window, min_periods=window).mean()
    )
    features_df[f"sales_std_{window}"] = (
        past_sales.rolling(window, min_periods=window).std()
    )

# Calendar information for the day being forecast.
features_df["weekday"] = features_df.index.dayofweek
features_df["month"] = features_df.index.month
features_df["is_weekend"] = (
    features_df.index.dayofweek >= 5
).astype(int)

# Remove initial rows without sufficient historical observations.
features_df = features_df.dropna().copy()

features_df.head()

,target_sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_mean_7,sales_std_7,sales_mean_28,sales_std_28,weekday,month,is_weekend
forecast_date,,,,,,,,,,,,
2009-12-29,0,0.0,14.0,36.0,204.0,2.142857,5.241774,83.642857,103.347388,1,12,0
2009-12-30,0,0.0,1.0,98.0,112.0,0.142857,0.377964,76.357143,101.726290,2,12,0
2009-12-31,0,0.0,0.0,90.0,134.0,0.000000,0.000000,72.357143,102.472115,3,12,0
2010-01-01,0,0.0,0.0,76.0,47.0,0.000000,0.000000,67.571429,102.615582,4,1,0
2010-01-02,0,0.0,0.0,0.0,12.0,0.000000,0.000000,65.892857,103.346357,5,1,1


In [13]:
# save product tabular dataset
features_df.to_csv(
    f"s3://{BUCKET}/smart-stock/data/modelling/"
    f"product_{PRODUCT_ID}_tabular_features.csv",
    index=True,
    index_label="forecast_date",
)